In [ ]:
# Phase 8 Screen B, fold 0: the label target. c2's six-slot config trained on steven_v4,
# on one head per label source, and on a probability blend of the three best sources;
# every arm scored against one fixed truth and on gold, and as a fifth member.
import glob, os, shutil, sys, time

GIT_SHA = 'cfde806'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.dataset import PreppedSlotDataset, _SLOT_PLANES
from knee.dicom import SLOTS, SLOTS_V3
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (ModelEma, differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# The stale-src guard, derived from what this run needs. Without packing the
# arms would differ in BatchNorm statistics as well as the table; without the
# v3 plane lookup the second arm dies on its first study.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel._slot_features', KneeModel._slot_features, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__,
     ('weights_df', 'label_columns', 'random_groups')),
    ('train_one_epoch', train_one_epoch, ('accumulation_steps', 'ema')),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, '_backbone_features'), 'rsna-knee-src predates slot packing'
import knee.infer, knee.reports
assert all(hasattr(knee.reports, n) for n in ('multi_source_labels', 'blend_sources')), (
    'rsna-knee-src predates the label-target builders')
assert hasattr(knee.infer, 'average_source_heads'), 'rsna-knee-src predates head averaging'
print('src carries packing, the group_attention head, random_groups and ModelEma | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# The runs: one session, three arms, c2's recipe throughout (efficientnet_b0, six v2
# slots, slot attention, Adam 1e-4 / OneCycle 3e-4, batch 8, 8 epochs, augment on, gold
# held out), the fixed five groups of the stored artifact. Only the training frame
# changes. The val loader carries the FIXED truth (steven_v4 binarised) for every arm.
from knee.infer import average_source_heads
from knee.reports import blend_sources, multi_source_labels

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = 'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,macro_auc,paired_delta,train_minutes,inference_seconds,promoted,gold_macro'
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

# _votes, from the folds-and-labels cell: ours, steven_v4, pilkwang_v2 (confidence
# folded), lixin73 (confidence folded), each indexed by StudyInstanceUID
SOURCES = ['ours_qwen3_4b', 'steven_v4', 'pilkwang_v2', 'lixin73']
assert len(_votes) == len(SOURCES), f'expected {len(SOURCES)} label sources, found {len(_votes)}'

def as_labels_df(frame):
    frame = frame.copy()
    frame.index.name = 'StudyInstanceUID'
    return frame.reset_index()

wide = as_labels_df(multi_source_labels(dict(zip(SOURCES, _votes)), LABEL_COLUMNS))
WIDE_COLS = [c for c in wide.columns if c != 'StudyInstanceUID']
assert len(WIDE_COLS) == 12 * len(SOURCES)
# the three best independent sources on the 58 gold studies; pilkwang (0.8142) is left out
blend = as_labels_df(blend_sources([_votes[1], _votes[0], _votes[3]], LABEL_COLUMNS))
print(f'wide frame {wide.shape}, blend target {blend.shape}; blend positive rate per label '
      f'{blend[LABEL_COLUMNS].mean().round(3).min()}-{blend[LABEL_COLUMNS].mean().round(3).max()} '
      f'(steven_v4 {LABEL_SOURCES["steven_v4"][0][LABEL_COLUMNS].mean().round(3).min()}-'
      f'{LABEL_SOURCES["steven_v4"][0][LABEL_COLUMNS].mean().round(3).max()})')

CONFIGS = [
    # name, training frame, its label columns, number of source heads
    ('t0_c2_steven', LABEL_SOURCES['steven_v4'][0], LABEL_COLUMNS, 1),
    ('t1_c2_multisrc', wide, WIDE_COLS, len(SOURCES)),
    ('t2_c2_blend3', blend, LABEL_COLUMNS, 1),
]
HYPOTHESIS = {
    't0_c2_steven':
        'Screen B control: c2 (six v2 slots, slot attention, efficientnet_b0, 8 epochs, packed) on '
        'steven_v4, the promoted target. Scored on steven_v4 binarised and on gold like every arm.',
    't1_c2_multisrc':
        't1: one head per label source (ours, steven_v4, pilkwang_v2, lixin73), 48 logits, masked '
        'BCE skipping the NaN where a source never rated a study; the four readings are averaged at '
        'inference. The multi-source target without the rank blend\'s destroyed prevalence.',
    't2_c2_blend3':
        't2: the probability mean of the three best independent sources on gold (steven_v4, ours, '
        'lixin73) as one soft target -- prevalence preserved, unlike the rank blend rejected in Phase 6.',
}
SOURCE, FOLD, N_GROUPS, OUT_SIZE, EPOCHS, BATCH = 'steven_v4', 0, 5, 224, 8, 8
TIME_BUDGET_S = 8 * 3600
t_start = time.time()

def loader(uids, labels_df, columns, shuffle=False, augment=False):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=N_GROUPS,
                            out_size=OUT_SIZE, slots=slot_names, augment=augment,
                            label_columns=columns)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=2)

_, eval_labels = LABEL_SOURCES[SOURCE]
tr_uids, va_uids = train_val_split(folds, val_fold=FOLD, exclude_uids=holdout)
tr_uids = [u for u in tr_uids if u in uid_to_npz]
va_uids = [u for u in va_uids if u in uid_to_npz]

results = []
for name, train_frame, columns, n_heads in CONFIGS:
    if time.time() - t_start > TIME_BUDGET_S:
        print(f'time budget spent, skipping {name}')
        break
    print(f'\n=== {name}: {len(tr_uids)} train / {len(va_uids)} val, {len(columns)} target columns, '
          f'{n_heads} head(s), {EPOCHS} epochs')
    train_loader = loader(tr_uids, train_frame, columns, shuffle=True, augment=True)
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(columns), pretrained=True,
                      head='slot_attention').to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4, epochs=EPOCHS, steps_per_epoch=len(train_loader))

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, device=device, scaler=scaler,
                               scheduler=scheduler)
        if epoch == 0:
            print(f'  peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
        print(f'  epoch {epoch + 1}/{EPOCHS} loss {loss:.4f} ({(time.time() - t0) / 60:.1f} min)')
    minutes = (time.time() - t0) / 60

    # the fixed truth rides on the val loader; the model's width is the arm's own
    y_true, heads = evaluate(model, loader(va_uids, eval_labels, LABEL_COLUMNS), device=device)
    g_true, g_heads = evaluate(model, loader(gold_uids, gold_labels, LABEL_COLUMNS), device=device)
    y_pred, g_pred = average_source_heads(heads, n_heads), average_source_heads(g_heads, n_heads)
    aucs = per_label_auc(y_true, y_pred)
    macro, gold_macro = macro_auc(y_true, y_pred), macro_auc(g_true, g_pred)
    print(f'  {name}: val macro {macro:.4f} (against steven_v4), GOLD macro {gold_macro:.4f} '
          f'in {minutes:.1f} min')
    if n_heads > 1:
        # each head alone, and the average without the weakest source on gold
        for k, src in enumerate(SOURCES):
            h, gh = heads[:, k * 12:(k + 1) * 12], g_heads[:, k * 12:(k + 1) * 12]
            print(f'    head {src:14s} val {macro_auc(y_true, h):.4f}  gold {macro_auc(g_true, gh):.4f}')
        keep = [k for k, src in enumerate(SOURCES) if src != 'pilkwang_v2']
        sub = np.mean([heads[:, k * 12:(k + 1) * 12] for k in keep], axis=0)
        gsub = np.mean([g_heads[:, k * 12:(k + 1) * 12] for k in keep], axis=0)
        print(f'    heads without pilkwang: val {macro_auc(y_true, sub):.4f}  gold {macro_auc(g_true, gsub):.4f}')
        np.save(f'/kaggle/working/oof_{name}_heads.npy', heads)
        np.save(f'/kaggle/working/gold_pred_{name}_heads.npy', g_heads)
    for label, auc in zip(LABEL_COLUMNS, aucs):
        print(f'    {label:20s} {auc:.4f}')
    results.append({'config': name, 'val_vs_steven': macro, 'gold': gold_macro, 'minutes': minutes,
                    'final_loss': loss})
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name, hypothesis=HYPOTHESIS[name],
                   fold_set=f'primary_v3_fold{FOLD}', seed=0,
                   per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(macro), paired_delta=float('nan'), train_minutes=minutes,
                   inference_seconds=float('nan'), promoted=False, gold_macro=float(gold_macro))
    np.save(f'/kaggle/working/oof_{name}.npy', y_pred)
    np.save(f'/kaggle/working/y_{name}.npy', y_true)
    np.save(f'/kaggle/working/gold_pred_{name}.npy', g_pred)
    np.save(f'/kaggle/working/gold_true_{name}.npy', g_true)
    torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

In [ ]:
# Replacement question (gold decides; the fixed-truth delta is reported, not obeyed),
# then each arm as a fifth member next to c2+c3+f1+s2 on gold and on the fixed truth.
from knee.metrics import paired_macro_auc_delta

def saved(kind, name):
    return np.load(f'/kaggle/working/{kind}_{name}.npy')

def mounted_one(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    assert len(hits) == 1, f'{pattern}: expected exactly one mounted file, found {hits}'
    return hits[0]

def same(a, b):
    return np.array_equal(np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1))

def show(label, y, a, b):
    d, lo, hi = paired_macro_auc_delta(y, a, b)
    verdict = 'CI excludes zero' if lo > 0 or hi < 0 else 'CI spans zero'
    print(f'  {label:52s} {d:+.4f} [{lo:+.4f}, {hi:+.4f}]  {verdict}')

print(pd.DataFrame(results)[['config', 'val_vs_steven', 'gold', 'final_loss', 'minutes']]
      .to_string(index=False))
done = [r['config'] for r in results]

CONFIRMED = ('c2_6slot_attn', 'c3_dinov2_2slot', 'f1_convnext_tiny_6slot_attn', 's2_c2_336_b4x2')

def confirm_fold(name):
    uids = pd.read_csv(mounted_one(f'oof_uids_{name}.csv'))['StudyInstanceUID'].astype(str)
    row = {u: i for i, u in enumerate(uids)}
    rows = [row[u] for u in va_uids]
    return (np.load(mounted_one(f'oof_true_{name}.npy'))[rows],
            np.load(mounted_one(f'oof_pred_{name}.npy'))[rows],
            np.load(mounted_one(f'gold_folds_{name}.npy'))[FOLD])

members = {n: confirm_fold(n) for n in CONFIRMED}
for tier, truth, pred, idx in (('gold', 'gold_true', 'gold_pred', 2), ('val (steven_v4 truth)', 'y', 'oof', 1)):
    if not done:
        break
    y = saved(truth, done[0])
    if idx == 1:
        assert all(same(y, members[n][0]) for n in CONFIRMED), 'members were scored on different studies'
    print(f'\n{tier} (n={len(y)}), against the control')
    for name in done[1:]:
        show(f'{name} minus t0_c2_steven', y, saved(pred, 't0_c2_steven'), saved(pred, name))
    four = np.mean([members[n][idx] for n in CONFIRMED], axis=0)
    print(f'{tier} (n={len(y)}), c2+c3+f1+s2 {macro_auc(y, four):.4f}; each arm as a fifth member '
          f'(bar +0.0015, CI above zero; t0 prices one more model of a known config)')
    for name in done:
        show(name, y, four, (4 * four + saved(pred, name)) / 5)
print(f'\ntotal {(time.time() - t_start) / 3600:.2f} h')